# UNN Singel-Snapshot DoA Sparse Array

[1] Y. Bu, J. Yu, and P. Pal, “Prediction-driven untrained network for single-snapshot sparse array interpolation,” in ICASSP 2025 - 2025 IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP), Apr. 2025, pp. 1–5.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import doatools.model as model
import doatools.performance as perf
import doatools.plotting as plt_tools
import doatools.estimation as estimation
%matplotlib inline

# set global font and size
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 18,
    'axes.titlesize': 18,
    'axes.labelsize': 18, 
    'legend.fontsize': 18,
    'legend.loc': 'lower right',
    'legend.title_fontsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18
})



## BaseLine

In [ ]:
from doatools.estimation import CovarianceReconstructionBase
from scipy.linalg import toeplitz
from scipy.optimize import least_squares


class NewtonianIdentitiesEstimator(CovarianceReconstructionBase):
    """Creates a source location estimator based on Newtonian Identities (NI) algorithm.

    The NI algorithm reconstructs the augmented covariance matrix using Newtonian identities
    to compute missing covariance elements from known ones.

    Args:
        array (~doatools.model.arrays.ArrayDesign): Array design.
        wavelength (float): Wavelength of the carrier wave.
        sigma_s2 (float, optional): Signal power. Defaults to 1.0.
        doa_estimator: DOA estimator instance. Defaults to None (RootMUSIC1D).
        search_grid (~doatools.estimation.grid.SearchGrid, optional): The search grid
            used to locate the sources. Defaults to None. Not needed for
            gridless DOA estimators like RootMUSIC1D and ESPRIT.
        **kwargs: Other keyword arguments.
    """

    def __init__(self, array, wavelength, sigma_s2: float = 1.0, doa_estimator=None, search_grid=None, **kwargs):
        super().__init__(array, wavelength, doa_estimator, search_grid, **kwargs)
        self._sigma_s2 = sigma_s2

    @staticmethod
    def _compute_elementary_symmetric_polynomials(p: np.ndarray, k: int) -> np.ndarray:
        """使用牛顿恒等式计算基本对称多项式 e_1, e_2, ..., e_k。

        Args:
            p: 已知的幂和数组 [p_1, p_2, ..., p_k]，支持复数。
            k: 变量个数。

        Returns:
            e: 基本对称多项式数组 [e_1, e_2, ..., e_k]。
        """
        e = np.zeros(k + 1, dtype=complex)
        e[0] = 1
        for n in range(1, k + 1):
            sum_term = 0
            for j in range(1, n + 1):
                if j - 1 < len(p):
                    sum_term += (-1) ** (j - 1) * e[n - j] * p[j - 1]
            e[n] = sum_term / n
        return e[1:]

    @staticmethod
    def _compute_power_sums_from_e(e: np.ndarray, k: int, m: int) -> np.ndarray:
        """使用牛顿恒等式和给定的e计算幂和 p_1 到 p_m。

        Args:
            e: 基本对称多项式数组 [e_1, e_2, ..., e_k]
            k: 变量个数
            m: 最大幂次

        Returns:
            p_array: 幂和数组 [p_1, p_2, ..., p_m]，索引从0开始对应p_1到p_m
        """
        p_array = np.zeros(m, dtype=complex)

        for n in range(1, m + 1):
            if n <= k:
                p_n = 0
                for j in range(1, n):
                    p_n += (-1) ** (j - 1) * e[j - 1] * p_array[n - j - 1]
                p_n += (-1) ** (n - 1) * n * e[n - 1]
            else:
                p_n = 0
                for j in range(1, k + 1):
                    p_n += (-1) ** (j - 1) * e[j - 1] * p_array[n - j - 1]
            p_array[n - 1] = p_n

        return p_array

    def _compute_missing_R(self, k: int, m: int, known_p: np.ndarray, known_indices: np.ndarray,
                           sigma_s2: complex = 1.0) -> np.ndarray:
        """计算缺失的 R_{n,1} 值。

        Args:
            k: 变量个数
            m: 方程总数（最大幂次）
            known_p: 已知的 R_{n,0} 值数组，形状为 (len(known_indices),)
            known_indices: 已知值的索引数组，形状为 (len(known_p),)
            sigma_s2: 信号功率的平方

        Returns:
            missing_R: 缺失的 R_{n,0} 值数组，形状为 (m,)
        """
        if len(known_indices) < k:
            raise ValueError(f"需要至少 {k} 个已知的 R值，但只提供了 {len(known_indices)} 个。")

        # 归一化已知值
        known_p_normalized = known_p / sigma_s2

        # 设置初始值e
        e_initial = np.zeros(k, dtype=complex)
        for i in range(k, 0, -1):
            if np.isin(np.arange(1, i + 1), known_indices).all():
                # 找到连续的索引
                mask = np.isin(known_indices, np.arange(1, i + 1))
                if np.sum(mask) == i:
                    selected_p = known_p_normalized[mask]
                    e_initial[:i] = self._compute_elementary_symmetric_polynomials(selected_p, i)
                    break

        # 将复数e转换为实数向量 [实部, 虚部]
        x0 = np.zeros(2 * k)
        x0[:k] = np.real(e_initial)
        x0[k:] = np.imag(e_initial)

        # 定义残差函数
        def residual_function(x: np.ndarray, known_p_norm: np.ndarray, known_idx: np.ndarray, k: int,
                              max_n: int) -> np.ndarray:
            """计算残差：预测值与已知值的差异"""
            # 将实数向量转换为复数e
            e_real = x[:k]
            e_imag = x[k:]
            e = e_real + 1j * e_imag

            # 使用当前e计算所有幂和
            p_array = self._compute_power_sums_from_e(e, k, max_n)

            residuals = []
            for i, n in enumerate(known_idx):
                if n > max_n:
                    continue
                pred_val = p_array[n - 1]
                true_val = known_p_norm[i]
                diff = (pred_val - true_val)
                residuals.append(np.real(diff))  # 实部残差
                residuals.append(np.imag(diff))  # 虚部残差

            return np.array(residuals)

        # 最大已知索引
        max_known = np.max(known_indices)

        # 使用最小二乘和Levenberg-Marquardt方法超定方程组
        res = least_squares(
            residual_function, x0,
            args=(known_p_normalized, known_indices, k, max_known),
            method='lm',
            ftol=1e-8,  # 函数容忍度
            xtol=1e-8,  # 参数容忍度
            max_nfev=1000,  # 最大函数评估次数
        )

        # 提取优化后的e
        x_opt = res.x
        e_opt = x_opt[:k] + 1j * x_opt[k:]

        # 使用优化后的e计算所有幂和
        p_array = self._compute_power_sums_from_e(e_opt, k, m)

        # 返回所有幂和（包括已知和缺失的）
        return sigma_s2 * p_array

    def reconstruct(self, R, **kwargs):
        """Reconstructs the augmented covariance matrix using Newtonian Identities algorithm.

        Args:
            R (~numpy.ndarray): Sample covariance matrix of the sparse array.

        Returns:
            ~numpy.ndarray: Augmented covariance matrix.
        """
        scm_vector = self._coarray_builder.transform(R, 'da')
        return scm_vector

    # Override the estimate method to use the correct NI algorithm flow
    def estimate(self, R, k, **kwargs):
        """Estimates the source locations from the given covariance matrix using NI algorithm.

        Args:
            R (~numpy.ndarray): Covariance matrix input. The size of R must
                match that of the array design used when creating this
                estimator.
            k (int): Expected number of sources.
            **kwargs: Other keyword arguments.

        Returns:
            A tuple with the following elements.

            * resolved (:class:`bool`): ``True`` if the desired number of
              sources are found. This flag does **not** guarantee that the
              estimated source locations are correct. The estimated source
              locations may be completely wrong!
              If resolved is False, both ``estimates`` and ``spectrum`` will be
              ``None``.
            * estimates (:class:`~doatools.model.sources.SourcePlacement`):
              A :class:`~doatools.model.sources.SourcePlacement` instance of the
              same type as the one used in the search grid (if provided),
              representing the estimated source locations. Will be ``None`` if resolved is
              ``False``.
            * spectrum (:class:`~numpy.ndarray`): An numpy array of the same
              shape of the specified search grid, consisting of values evaluated
              at the grid points. Only present if ``return_spectrum`` is
              ``True`` and a search grid is provided.
        """
        n, m = self._S.shape
        scm_vector = self.reconstruct(R)[:, 0]
        non_zero_mask = scm_vector[1:] != 0.0
        known_indices = np.where(non_zero_mask)[0] + 1
        known_values = scm_vector[known_indices]
        try:
            all_R_values = self._compute_missing_R(k, m - 1, known_values, known_indices, self._sigma_s2)
            scm_vector[1:m] = all_R_values
            Ra = toeplitz(scm_vector)
        except Exception as e:
            print(f"NI algorithm failed: {e}")
            Ra = toeplitz(scm_vector)
            Ra = (Ra + Ra.conj().T) / 2
        return self._doa_estimate(Ra, k, **kwargs)



## 1. Untrain Network Algorithm Implementation

In [ ]:
from matplotlib import axis
from doatools.estimation import CovarianceReconstructionBase
from scipy.linalg import toeplitz, hankel, lstsq
from scipy.optimize import least_squares
import warnings

class UntrainNNEstimator(CovarianceReconstructionBase):
    """
    Creates a source location estimator based on Prediction-driven Untrained Network (UNN) 
    for Single-snapshot Sparse Array Interpolation.

    This method models the single-snapshot signal interpolation as a linear prediction problem 
    governed by latent variables 'm'. It optimizes 'm' to minimize prediction error on observed 
    sparse sensors while maintaining consistency on the dense ULA segment.

    Args:
        array (~doatools.model.arrays.GridBasedArrayDesign): Array design (must be 1D grid-based).
        wavelength (float): Wavelength of the carrier wave.
        n_sources (int): Number of sources (K). Required for the algorithm.
        doa_estimator: DOA estimator instance. Defaults to None.
        search_grid: Search grid. Defaults to None.
        **kwargs: Other keyword arguments.
    """

    def __init__(self, array, wavelength, n_sources, doa_estimator=None, search_grid=None, **kwargs):
        super().__init__(array, wavelength, doa_estimator, search_grid, **kwargs)
        self._K = n_sources
        # 1. 获取网格索引 (Integer Indices)
        indices = array.element_indices
        # 扁平化并确保是整数
        self._pos_indices = indices.flatten().astype(int)
        # 3. 确定孔径和虚拟 ULA 索引
        self._aperture = self._pos_indices.max() + 1
        self._U_indices = np.arange(self._aperture)
        # 4. 识别 S1 (密集部分) 和 S2 (稀疏部分)
        # S1 定义为从 0 开始的最长连续整数序列: {0, 1, ..., M_end}
        diffs = np.diff(self._pos_indices)
        # 找到步长大于 1 的位置
        break_points = np.where(diffs > 1)[0]
        if len(break_points) > 0:
            # S1 结束于第一个断点之前
            s1_end_idx = break_points[0] + 1
        else:
            # 如果没有断点，说明整个阵列都是 ULA
            s1_end_idx = len(self._pos_indices)
        # 记录物理阵列中属于 S1 和 S2 的**索引位置** (对应 y 向量的下标)
        # 例如: y = [y0, y1, y2, y7, y11], S1_indices 对应 [0, 1, 2], S2 对应 [3, 4]
        # s1_end_idx -= 1
        self._S1_len = s1_end_idx
        # 记录 S2 在**虚拟网格**中的绝对位置 (用于计算残差)
        # 例如: S2 的网格位置可能是 [7, 11]
        self._S2_grid_indices = self._pos_indices[s1_end_idx:]
        # 验证 Lemma 2 条件: M + 1 >= 2K, S1 的长度必须足够大以初始化线性预测
        if self._S1_len < 2 * self._K:
            warnings.warn(
                f"Dense segment size ({self._S1_len}) is smaller than 2K ({2*self._K}). "
                "Initialization (Lemma 2) might be unstable or invalid."
            )

    def reconstruct(self, R, Y=None, **kwargs):
        """
        Reconstructs the augmented covariance matrix using UNN algorithm.

        Args:
            R (~numpy.ndarray): Sample covariance matrix.
            Y (~numpy.ndarray, optional): Single snapshot signal (P x 1). 
                                          If None, extracted from R.

        Returns:
            ~numpy.ndarray: Interpolated covariance matrix (N x N).
        """
        # 1. 准备信号向量 y
        if Y is None:
            print("ERROR: Y is None")
        else:
            Y = Y.mean(axis=1)
            y_obs = Y.flatten()

        K = self._K
        
        # 分离 S1 和 S2 的观测数据
        # S1 是观测向量的前面连续部分
        y_S1 = y_obs[:self._S1_len] 
        y_S2 = y_obs[self._S1_len:] 

        # 2. 初始化 m (基于 Lemma 2)
        # 使用 S1 数据构建 Hankel 矩阵求解 m
        L_S1 = len(y_S1)
        if L_S1 > K:
            # 构建 Hankel 矩阵: cols = K, rows = L_S1 - K
            # H * m = y_target
            # H 的列: [y(0)...y(L-K-1)]^T, ...
            # y_target: [y(K)...y(L-1)]^T
            
            # 构造数据矩阵 X 和目标向量 d
            # y_n = \sum_{i=1}^K m_i * y_{n-K-1+i}
            # 这意味着 m_1 对应最旧的样本 y_{n-K}
            
            # 使用 scipy.linalg.hankel 构造
            # 第一列: y_S1[0 : L-K]
            # 最后一列: y_S1[K-1 : L-1]
            c = y_S1[:L_S1-K]
            r = y_S1[L_S1-K-1:-1] # 注意切片范围，hankel函数需要最后一行的最后一个元素
            H_mat = hankel(c, r)
            
            target_vec = y_S1[K:]
            
            # 最小二乘求解 m0
            m0, _, _, _ = lstsq(H_mat, target_vec)
        else:
            # 随机初始化 (备用)
            m0 = np.random.randn(K) + 1j * np.random.randn(K)
            m0 = m0 / np.linalg.norm(m0)

        # 3. 优化过程 (Untrained Learning)
        # 目标: min L(m) = L1(S1一致性) + L2(S2预测误差)
        
        # 定义残差函数 (输入为 [real(m), imag(m)])
        res_func = lambda m_flat: self._compute_residuals(m_flat, y_S1, y_S2, self._S2_grid_indices, K)
        
        m0_flat = np.concatenate([m0.real, m0.imag])
        
        # 使用 Levenberg-Marquardt (lm) 求解非线性最小二乘
        result = least_squares(res_func, m0_flat, method='lm')
        
        m_opt_flat = result.x
        m_opt = m_opt_flat[:K] + 1j * m_opt_flat[K:]

        # 4. 生成完整信号并构建协方差矩阵
        # 使用最优 m 生成整个虚拟孔径的数据
        y_U = self._generate_signal(m_opt, y_S1, self._aperture, K)
        # y_U = y_U.reshape(-1, 1)
        # 构建Y和Y的协方差矩阵
        # R_rec = y_U @ y_U.conj().T
        R_rec = np.outer(y_U, y_U.conj())
        
        return R_rec

    def _generate_signal(self, m, y_S1, total_len, K):
        """
        Generates the full signal using the recursive linear model.
        """
        y_full = np.zeros(total_len, dtype=complex)
        
        # S1 部分直接使用观测值 (作为 Ground Truth)
        L_S1 = len(y_S1)
        y_full[:L_S1] = y_S1
        
        # 递归预测剩余部分
        # y_n = \sum_{i=1}^K m_i * y_{n-K-1+i}
        # m 向量对应 [y_{n-K}, y_{n-K+1}, ..., y_{n-1}] 的系数
        
        for n in range(L_S1, total_len):
            # 取前 K 个样本作为 context
            context = y_full[n-K : n]
            # 计算点积 (注意 m 的顺序与 context 的顺序对应)
            # context: [y_{n-K}, ..., y_{n-1}]
            # m: [m_1, ..., m_K]
            val = np.dot(context, m) 
            y_full[n] = val
            
        return y_full

    def _compute_residuals(self, m_flat, y_S1, y_S2, S2_grid_indices, K):
        """
        Computes the residual vector for least_squares.
        """
        m = m_flat[:K] + 1j * m_flat[K:]
        
        # 1. 在 S1 内部的线性预测误差 (L1 Loss)
        # 衡量 m 是否能很好地拟合已知的 S1 数据结构
        L_S1 = len(y_S1)
        if L_S1 > K:
            # 同样构造 Hankel 结构进行批量预测
            c = y_S1[:L_S1-K]
            r = y_S1[L_S1-K-1:-1]
            H_mat = hankel(c, r)
            preds_S1 = H_mat @ m
            res_S1 = preds_S1 - y_S1[K:]
        else:
            res_S1 = np.array([])
        
        # 2. 在 S2 处的预测误差 (L2 Loss)
        # 生成延伸到 S2 最远处的信号
        max_idx = S2_grid_indices.max() if len(S2_grid_indices) > 0 else L_S1
        y_gen_full = self._generate_signal(m, y_S1, max_idx + 1, K)
        
        if len(S2_grid_indices) > 0:
            # 取出生成信号中对应 S2 网格位置的值
            preds_S2 = y_gen_full[S2_grid_indices]
            res_S2 = preds_S2 - y_S2
        else:
            res_S2 = np.array([])
        
        # 合并残差并分离实虚部
        full_res_complex = np.concatenate([res_S1, res_S2])
        return np.concatenate([full_res_complex.real, full_res_complex.imag])

## DOA 估计

In [ ]:
# Basic parameters
wavelength = 1.0
d0 = wavelength / 2
source_angles = np.deg2rad(np.array([10, 20]))
n_sources = 2

array_nest = model.GridBasedArrayDesign(indices=np.array([0, 1, 2, 3, 4, 9, 14, 19])[:, np.newaxis], d0=d0, name='nest array')
array_nest_ula = model.GridBasedArrayDesign(indices=np.arange(20)[:, np.newaxis], d0=d0, name='nest array ULA')
array_ula8 = model.GridBasedArrayDesign(indices=np.array([0, 1, 2, 3, 4, 5, 6, 7])[:, np.newaxis], d0=d0, name='ULA-8')
print(f'Number of physical elements: {array_nest.size}')
print(f'Number of physical elements: {array_ula8.size}')

# Simulation parameters
n_snapshots = 1
n_snr = 30
n_monte_carlo = 500

# Create the estimators
estimators_nest = {
    'DA+MUSIC': estimation.DAEstimator(array_nest, wavelength),
    # 'ANM+MUSIC': estimation.ANMEstimator(array_nest, wavelength),
    'UNN+MUSIC': UntrainNNEstimator(array_nest, wavelength,n_sources),
    'NI+MUSIC':  NewtonianIdentitiesEstimator(array_nest, wavelength)
}

# Create the estimators
estimators_ula8 = {
    'ULA8+MUSIC': estimation.CovarianceReconstructionBase(array_ula8, wavelength)
}


In [ ]:
delta_values = np.arange(3, 11) # delta 取值范围是 3,4,5,6,7,8,9,10
# Initialize results storage
delta_results = {alg: {'mse': [], 'bias': [], 'PR': []} for alg in estimators_nest.keys()}
delta_results.update({alg: {'mse': [], 'bias': [], 'PR': []} for alg in estimators_ula8.keys()})
crb_delta_values = {"nest": [], "ula8": []}
# Run simulation for each delta
for delta in delta_values:
    source_angles = np.deg2rad(np.array([10, 10+delta]))
    sources = model.FarField1DSourcePlacement(source_angles)
    print(f"delta -> {delta}")
    # Run performance evaluation
    result_nest = perf.evaluate_performance(
        array=array_nest,
        sources=sources,
        snr=n_snr,
        n_snapshots=n_snapshots,
        n_monte_carlo=n_monte_carlo,
        estimators=estimators_nest,
        crb_types=['stouc'],
        metrics=['mse', 'bias'],
        save_sample_estimates=True,
        verbose=1,
        n_jobs=1,
    )
    result_ula8 = perf.evaluate_performance(
        array=array_ula8,
        sources=sources,
        snr=n_snr,
        n_snapshots=n_snapshots,
        n_monte_carlo=n_monte_carlo,
        estimators=estimators_ula8,
        crb_types=['stouc'],
        metrics=['mse', 'bias'],
        save_sample_estimates=True,
        verbose=1,
        n_jobs=1,
    )

    # Store results
    for alg in estimators_nest.keys():
        delta_results[alg]['mse'].append(result_nest.estimator_results[alg]['mse'])
        delta_results[alg]['bias'].append(result_nest.estimator_results[alg]['bias'])
        samples = result_nest.sample_estimates[alg]
        # 计算PR
        PR = np.mean(np.abs(samples - sources.locations) < np.deg2rad(delta/2))
        delta_results[alg]['PR'].append(PR)
    crb_delta_values["nest"].append(result_nest.crb_values['stouc'])
    for alg in estimators_ula8.keys():
        delta_results[alg]['mse'].append(result_ula8.estimator_results[alg]['mse'])
        delta_results[alg]['bias'].append(result_ula8.estimator_results[alg]['bias'])
        samples = result_ula8.sample_estimates[alg]
        # 计算PR
        PR = np.mean(np.abs(samples - sources.locations) < np.deg2rad(delta/2))
        delta_results[alg]['PR'].append(PR)
    crb_delta_values['ula8'].append(result_ula8.crb_values['stouc'])


# Convert lists to numpy arrays for easier plotting
for alg in delta_results.keys():
    delta_results[alg]['mse'] = np.array(delta_results[alg]['mse'])
    delta_results[alg]['bias'] = np.array(delta_results[alg]['bias'])
crb_delta_values['nest'] = np.array(crb_delta_values['nest'])
crb_delta_values['ula8'] = np.array(crb_delta_values['ula8'])

print('delta performance test completed!')


In [ ]:
# Plot MSE vs SNR using our plotting tool
fig, ax = plt.subplots(figsize=(10, 8))
plt_tools.plot_metric_vs_parameter(
    parameter_values=delta_values,
    results={alg: delta_results[alg]['PR'] for alg in delta_results.keys()},
    parameter_name=rf'Angle Separation $\Delta$',
    metric_name='PR',
    parameter_unit='',
    metric_unit='',
    show_crb=False,
    ax=ax
    )
ax.set_ylim(0, 1)
ax.set_yscale('linear')
plt.title("Resolution Probability")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import doatools.model as model
import doatools.estimation as estimation

# === 1. 参数设置 ===
angle_separations = [10, 4]
snr_list = [10, 20, 30]
colors = {10: 'red', 20: 'blue', 30: 'green'}
grid = estimation.FarField1DSearchGrid(size=1800)

# 初始化估计器
music_nest = estimation.MUSIC(array_nest_ula, wavelength, grid)
music_ula8 = estimation.MUSIC(array_ula8, wavelength, grid)
da_estimator = estimation.DAEstimator(array_nest, wavelength)
anm_estimator = estimation.ANMEstimator(array_nest, wavelength)
unn_estimator = UntrainNNEstimator(array_nest, wavelength, n_sources)
ni_estimator = NewtonianIdentitiesEstimator(array_nest, wavelength, 1.0)

# 创建画布 2行3列
fig, axes = plt.subplots(2, 5, figsize=(20, 10))

# === 2. 主循环：遍历角度间隔 ===
for row_idx, sep in enumerate(angle_separations):
    # 当前角度配置
    angles = np.array([10, 10 + sep])
    source_angles = np.deg2rad(angles)
    sources = model.FarField1DSourcePlacement(source_angles)
    # 用于存储当前行、不同SNR下的谱估计结果
    res_da = {}
    res_unn = {}
    res_anm = {}
    res_ni = {}
    res_ula = {}
    # === 3. 内层循环：遍历SNR ===
    for snr in snr_list:
        power_source = 1.0
        power_noise = power_source * 10**(-snr/10.0)
        # 生成信号 (保证每次随机性，或者固定种子)
        source_signal = model.ComplexStochasticSignal(sources.size, power_source)
        # --- Nested Array Data ---
        noise_signal_nest = model.ComplexStochasticSignal(array_nest.size, power_noise)
        Y_nest, R_nest = model.get_narrowband_snapshots(
            array_nest, sources, wavelength, source_signal, noise_signal_nest,
            n_snapshots, return_covariance=True
        )
        # --- ULA8 Data ---
        noise_signal_ula = model.ComplexStochasticSignal(array_ula8.size, power_noise)
        _, R_ula8 = model.get_narrowband_snapshots(
            array_ula8, sources, wavelength, source_signal, noise_signal_ula,
            n_snapshots, return_covariance=True
        )
        # --- 算法 0: DA ---
        R_da = da_estimator.reconstruct(R_nest)
        _, _, sp_da = music_nest.estimate(R_da, sources.size, return_spectrum=True)
        res_da[f'SNR = {snr}dB'] = sp_da
        # --- 算法 1: UNN ---
        R_unn = unn_estimator.reconstruct(None, Y_nest) 
        _, _, sp_unn = music_nest.estimate(R_unn, sources.size, return_spectrum=True)
        res_unn[f'SNR = {snr}dB'] = sp_unn
        # --- 算法 2: Nuclear Norm (ANM) ---
        R_anm = anm_estimator.reconstruct(R_nest)
        _, _, sp_anm = music_nest.estimate(R_anm, sources.size, return_spectrum=True)
        res_anm[f'SNR = {snr}dB'] = sp_anm
        # --- 算法 3: NI (NI) ---
        R_ni = ni_estimator.reconstruct(R_nest)
        _, _, sp_ni = music_nest.estimate(R_ni, sources.size, return_spectrum=True)
        res_ni[f'SNR = {snr}dB'] = sp_ni
        # --- 算法 3: Standard MUSIC on ULA ---
        _, _, sp_ula8 = music_ula8.estimate(R_ula8, sources.size, return_spectrum=True)
        res_ula[f'SNR = {snr}dB'] = sp_ula8

    # === 4. 当前行绘图 ===
    # 辅助函数：自定义绘图以匹配颜色
    def plot_custom(ax, result_dict, title):
        for snr_val in snr_list:
            label = f'SNR = {snr_val}dB'
            if label in result_dict:
                sp = result_dict[label]
                sp_log = 10 * np.log10(sp / np.max(sp))
                ax.plot(np.rad2deg(grid.axes)[0], sp_log, 
                        label=label, color=colors[snr_val], linewidth=1.5)
        # 绘制 Ground Truth 垂直线
        for angle in angles:
            ax.axvline(x=angle, color='skyblue', linestyle='-', linewidth=2, alpha=0.7)
        ax.set_title(title)
        ax.set_xlim([0, 30]) # 根据图片限制 x 轴
        ax.set_ylim([-60, 0]) # 根据图片限制 y 轴
        ax.set_xlabel("Angle in degree")
        ax.set_ylabel("pseudo spectrum (dB)")
        ax.grid(True, linestyle=':', alpha=0.6)
        # 仅在第一列显示图例，或者全部显示
        if row_idx == 0 and "ULA" in title: 
             ax.legend(loc='lower right', fontsize='small')

    # 第一列: UNN
    plot_custom(axes[row_idx, 0], res_unn, f"Our approach on Nested8 (Sep={sep})")
    # 第二列: DA
    plot_custom(axes[row_idx, 1], res_da, f"DA on Nested8 (Sep={sep})")
    # 第三列: ANM
    plot_custom(axes[row_idx, 2], res_anm, f"ANM on Nested8 (Sep={sep})")
    # 第四列: ANM
    plot_custom(axes[row_idx, 3], res_ni, f"NI on Nested8 (Sep={sep})")
    # 第五列: MUSIC ULA
    plot_custom(axes[row_idx, 4], res_ula, f"MUSIC on ULA8 (Sep={sep})")

plt.tight_layout()
plt.show()